# StayWise — Hotel Recommendation System

**Milestone 1: dataset loading, validation, and exploratory analysis.** This notebook uses ViHoRec, a research dataset of Vietnamese hotels with pseudonymous user–hotel ratings and dates. Source: https://github.com/MinhNguyenDS/ViHoRec. The dataset is released under CC BY-NC 4.0; follow its license and cite the dataset descriptor (https://arxiv.org/abs/2607.12946).

The hotel catalogue has limited attributes (hotel name and location), so content-based recommendation will be constrained. The dataset does not provide live room availability or current prices.

## 1. Environment and reproducibility
Run in Google Colab. Data is loaded directly from the dataset's public GitHub release; it is not copied into this project repository.

In [ ]:
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)

print('Python libraries ready; random seed =', SEED)


## 2. Load the released data
The source tables are small enough for a first Colab audit. If the source repository changes, verify the files and license before proceeding.

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/MinhNguyenDS/ViHoRec/Master/release'
interactions = pd.read_csv(f'{BASE_URL}/interactions.csv')
hotels = pd.read_csv(f'{BASE_URL}/hotels.csv')

print('Interactions shape:', interactions.shape)
print('Hotels shape:', hotels.shape)
display(interactions.head())
display(hotels.head())


## 3. Validate schema and data quality

In [ ]:
required_interaction_cols = {'user_id', 'hotel_id', 'rating', 'date', 'source'}
required_hotel_cols = {'hotel_id', 'name', 'location'}
assert required_interaction_cols.issubset(interactions.columns), f'Missing interaction columns: {required_interaction_cols - set(interactions.columns)}'
assert required_hotel_cols.issubset(hotels.columns), f'Missing hotel columns: {required_hotel_cols - set(hotels.columns)}'

interactions['date'] = pd.to_datetime(interactions['date'], errors='coerce')
interactions['rating'] = pd.to_numeric(interactions['rating'], errors='coerce')

print('Interaction dtypes:')
display(interactions.dtypes.astype(str).to_frame('dtype'))
print('Missing values — interactions:')
display(interactions.isna().sum().to_frame('missing_count'))
print('Missing values — hotels:')
display(hotels.isna().sum().to_frame('missing_count'))
print('Exact duplicate interactions:', int(interactions.duplicated().sum()))
print('Duplicate hotel IDs:', int(hotels['hotel_id'].duplicated().sum()))
print('Rating min/max:', interactions['rating'].min(), interactions['rating'].max())
print('Date range:', interactions['date'].min(), 'to', interactions['date'].max())


## 4. Exploratory analysis

In [ ]:
summary = pd.DataFrame({
    'value': [len(interactions), interactions['user_id'].nunique(), interactions['hotel_id'].nunique(), interactions['rating'].mean(), interactions['rating'].median(), interactions['source'].nunique()],
}, index=['interaction_rows', 'unique_users', 'unique_hotels_in_interactions', 'mean_rating', 'median_rating', 'sources'])
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
interactions['rating'].dropna().plot(kind='hist', bins=20, ax=axes[0], title='Rating distribution')
interactions['source'].fillna('Missing').value_counts().plot(kind='bar', ax=axes[1], title='Interactions by source')
axes[1].set_ylabel('Rows')
plt.tight_layout()
plt.show()

user_counts = interactions['user_id'].value_counts()
item_counts = interactions['hotel_id'].value_counts()
print('Users with >= 2 interactions:', int((user_counts >= 2).sum()))
print('Hotels with >= 5 interactions:', int((item_counts >= 5).sum()))
display(user_counts.describe().to_frame('interactions_per_user'))
display(item_counts.describe().to_frame('interactions_per_hotel'))


## 5. Join the hotel catalogue
Keep the interaction table and hotel table conceptually separate. This join adds the available hotel name and location to each interaction; it does not add prices, amenities, or live availability.

In [ ]:
data = interactions.merge(hotels, on='hotel_id', how='left', validate='many_to_one')
print('Merged shape:', data.shape)
print('Interactions without a matched hotel record:', int(data['name'].isna().sum()))
display(data.head())
display(hotels['location'].value_counts(dropna=False).head(15).to_frame('hotel_count'))


## 6. Five preprocessing techniques — audit and preparation

These are the planned study dimensions. The notebook records a minimally processed baseline and a cleaned variant. Valid extreme ratings must not be discarded merely because they are unusual. Learned encoders and TF-IDF vectorizers will be fitted only on training data in the modeling milestone.

In [ ]:
# Technique 1 — missing values: ratings and identifiers are essential for interaction evaluation.
# Keep a record of exclusions; do not silently impute missing target ratings.
before_rows = len(data)
required_fields = ['user_id', 'hotel_id', 'rating', 'date']
clean = data.dropna(subset=required_fields).copy()
missing_required_removed = before_rows - len(clean)

# Technique 2 — duplicate checks: remove exact duplicate interaction rows.
before_dedup = len(clean)
clean = clean.drop_duplicates(subset=['user_id', 'hotel_id', 'rating', 'date', 'source']).copy()
duplicates_removed = before_dedup - len(clean)

# Technique 3 — domain-aware rating validation: ratings outside the observed 1–10 scale are invalid.
invalid_rating_mask = ~clean['rating'].between(1, 10, inclusive='both')
invalid_rating_rows = int(invalid_rating_mask.sum())
clean = clean.loc[~invalid_rating_mask].copy()

# Technique 4 — categorical features: normalize missing source/location values for later encoding.
clean['source'] = clean['source'].fillna('Unknown').astype(str).str.strip()
clean['location'] = clean['location'].fillna('Unknown').astype(str).str.strip()

# Technique 5 — conservative text normalization for hotel names and locations.
def normalize_text(value):
    if not isinstance(value, str):
        return ''
    value = value.lower().strip()
    value = re.sub(r'[^\w\sÀ-ỹ-]', ' ', value, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', value).strip()

hotels_clean = hotels.drop_duplicates(subset=['hotel_id']).copy()
hotels_clean['name_clean'] = hotels_clean['name'].map(normalize_text)
hotels_clean['location_clean'] = hotels_clean['location'].map(normalize_text)
hotels_clean['content_text'] = (hotels_clean['name_clean'] + ' ' + hotels_clean['location_clean']).str.strip()

print('Rows before required-field filtering:', before_rows)
print('Rows removed for missing required fields:', missing_required_removed)
print('Exact duplicate interaction rows removed:', duplicates_removed)
print('Invalid rating rows removed:', invalid_rating_rows)
print('Rows in cleaned interactions:', len(clean))
print('Unique hotels after catalogue deduplication:', hotels_clean['hotel_id'].nunique())
display(clean.head())
display(hotels_clean.head())


## 7. Candidate split for ranking experiments

The dataset has timestamps, so the modeling phase will use a chronological holdout rather than randomly mixing future interactions into training. Before final evaluation, specify the relevance threshold, candidate set, and rules for users with too little history. A rating threshold such as 7.0 is a candidate only; inspect the rating distribution and justify it before reporting results.

Do not interpret an EDA or split summary as model performance.

In [ ]:
RATING_RELEVANCE_CANDIDATE = 7.0
print('Candidate positive relevance threshold:', RATING_RELEVANCE_CANDIDATE)
print('Interactions at/above candidate threshold:', int((clean['rating'] >= RATING_RELEVANCE_CANDIDATE).sum()))
print('Users with at least 3 cleaned interactions:', int((clean['user_id'].value_counts() >= 3).sum()))

# Example chronological per-user holdout preview; this is a diagnostic, not the final evaluation protocol.
ordered = clean.sort_values(['user_id', 'date', 'hotel_id'])
last_per_user = ordered.groupby('user_id', as_index=False).tail(1)
print('Users with a last interaction at/above candidate threshold:', int((last_per_user['rating'] >= RATING_RELEVANCE_CANDIDATE).sum()))
display(last_per_user[['user_id', 'hotel_id', 'rating', 'date']].head())


## 8. Next implementation milestone

After reviewing the audit output, implement the five recommendation approaches with one shared train/validation/test protocol: (1) popularity baseline, (2) content-based TF-IDF ranking, (3) user-based collaborative filtering, (4) matrix factorization, and (5) hybrid ranking. Compare minimally processed and processed variants without data leakage. Report NDCG@10, Precision@10, Recall@10, and Hit Rate@10 where defined, plus coverage and latency if feasible. Fill result tables only with actual measured scores.